# Evaluate Your SageMaker Endpoint with Inspect AI

## What is Inspect AI?

[Inspect AI](https://inspect.ai-safety-institute.org.uk/) is an open-source framework for LLM evaluations by the UK AI Safety Institute. It provides a standardized way to define benchmarks, run them against any model, and score the results — making evaluations reproducible and comparable across models.

## What is the SageMaker Inspect AI Container?

The [SageMaker Inspect AI container](https://docs.aws.amazon.com/nova/latest/userguide/inspect-ai-sagemaker-eval.html) runs LLM evaluations as a SageMaker Training Job. It uses the Training Job as managed compute for orchestration (not for actual model training) — your model runs separately on its own inference infrastructure.

## How This Notebook Works

Evaluate an existing SageMaker endpoint using two approaches:

| Approach | How it runs | Best for |
|----------|------------|----------|
| **Local SDK** | `inspect eval` CLI on your machine | Fast iteration, development |
| **Container Job** | SageMaker Training Job | Scalable, hands-off, production |

Supports both **SFT** (instruction-tuned, chat-capable) and **CPT** (base/pre-trained, completion only) models.

## Prerequisites

1. **A running SageMaker endpoint** — deployed with a Nova model or OpenAI-compatible server
2. **AWS credentials configured** — [Configuring the AWS CLI](https://docs.aws.amazon.com/cli/latest/userguide/cli-chap-configure.html)
3. **Python 3.10+**

**Time to complete:** ~5-10 minutes | **Cost:** Your endpoint's existing cost + `ml.m5.large` (~$0.13/hr) for Approach 2

## Configuration

In [ ]:
# =============================================================================
# UPDATE THESE VALUES
# =============================================================================

ENDPOINT_NAME = "your-endpoint-name"  # <-- REPLACE (e.g., nova-lite-2-endpoint)
REGION = "us-east-1"                  # Must match your endpoint's region

# Model type: "sft" or "cpt"
#   SFT = instruction-tuned, chat-capable (default, e.g., Nova Lite 2)
#   CPT = base/pre-trained, completion only (requires completion_mode)
MODEL_TYPE = "sft"

print(f"Endpoint: {ENDPOINT_NAME}")
print(f"Region:   {REGION}")
assert MODEL_TYPE in ("sft", "cpt"), f"MODEL_TYPE must be sft or cpt, got: {MODEL_TYPE}"
print(f"Type:     {MODEL_TYPE}")


### Verify Your Endpoint

In [ ]:
import boto3

sagemaker_client = boto3.client("sagemaker", region_name=REGION)
try:
    resp = sagemaker_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
    status = resp["EndpointStatus"]
    if status == "InService":
        print(f"\u2713 Endpoint '{ENDPOINT_NAME}' is InService")
    else:
        print(f"\u26a0 Endpoint status: {status}. Wait for InService before proceeding.")
except sagemaker_client.exceptions.ClientError as e:
    if "Could not find endpoint" in str(e):
        print(f"\u2717 Endpoint '{ENDPOINT_NAME}' not found in {REGION}.")
        print(f"  List endpoints: aws sagemaker list-endpoints --region {REGION}")
    else:
        raise

---

## Approach 1: Local SDK (Quick Evaluation)

Run evaluations directly on your machine. Results in seconds for small sample sizes.

**Benchmarks:**
- **SFT models:** MMLU-Pro (multiple-choice reasoning)
- **CPT models:** ARC-Challenge (science reasoning, completion mode)

In [ ]:
%pip install "inspect-ai>=0.3.220" inspect-evals aioboto3 openai --quiet

### Run Evaluation

In [ ]:
import subprocess

# Build the command based on model type
if MODEL_TYPE == "sft":
    cmd = f"inspect eval inspect_evals/mmlu_pro --model sagemaker/{ENDPOINT_NAME} -M region_name={REGION} --limit 5 --max-connections 4 --max-retries 3 --display plain"
else:
    cmd = f"inspect eval inspect_evals/arc_challenge --model sagemaker/{ENDPOINT_NAME} -M region_name={REGION} -M completion_mode=True --limit 5 --max-connections 4 --max-retries 3 --display plain"

print(f"Running: {cmd}\n")
subprocess.run(cmd, shell=True)

In [ ]:
!inspect view

---

## Approach 2: Container Job (Scalable Evaluation)

Run the same evaluation as a SageMaker Training Job. The container:
1. Downloads your config and benchmarks from S3
2. Installs benchmark dependencies
3. Sends requests to your endpoint for each sample
4. Publishes results to S3 incrementally

No GPU needed — the `ml.m5.large` orchestrator only coordinates the eval.

In [ ]:
%pip install "boto3>=1.35" "sagemaker>=3.0.0" pyyaml --quiet

### Setup: IAM Role and S3 Bucket

In [ ]:
import json, os, time, yaml
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import InputData, Compute
from sagemaker.core.shapes.shapes import StoppingCondition, OutputDataConfig

ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
S3_BUCKET = f"inspectlens-eval-{ACCOUNT_ID}"
ROLE_NAME = "InspectLensEvalRole"
ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}"
IMAGE_URI = f"763104351884.dkr.ecr.{REGION}.amazonaws.com/sagemaker-inspect-ai:latest"

# Create IAM role
iam = boto3.client("iam")
trust_policy = {"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Principal": {"Service": "sagemaker.amazonaws.com"}, "Action": "sts:AssumeRole"}]}
try:
    iam.create_role(RoleName=ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust_policy))
    print(f"\u2713 Created role: {ROLE_NAME}")
except iam.exceptions.EntityAlreadyExistsException:
    print(f"\u2713 Role exists: {ROLE_NAME}")
policies = ["arn:aws:iam::aws:policy/AmazonSageMakerFullAccess", "arn:aws:iam::aws:policy/AmazonS3FullAccess"]
for arn in policies:
    iam.attach_role_policy(RoleName=ROLE_NAME, PolicyArn=arn)
# Add MLflow permissions (for optional tracking)
iam.put_role_policy(RoleName=ROLE_NAME, PolicyName="MLflowAccess", PolicyDocument=json.dumps({"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Action": ["sagemaker:DescribeMlflowTrackingServer", "sagemaker:CreatePresignedMlflowTrackingServerUrl", "sagemaker-mlflow:*"], "Resource": f"arn:aws:sagemaker:{REGION}:{ACCOUNT_ID}:mlflow-tracking-server/*"}]}))
time.sleep(10)

# Create S3 bucket
s3 = boto3.client("s3", region_name=REGION)
try:
    s3.create_bucket(Bucket=S3_BUCKET) if REGION == "us-east-1" else s3.create_bucket(Bucket=S3_BUCKET, CreateBucketConfiguration={"LocationConstraint": REGION})
    print(f"\u2713 Created bucket: {S3_BUCKET}")
except s3.exceptions.BucketAlreadyOwnedByYou:
    print(f"\u2713 Bucket exists: {S3_BUCKET}")

### Create and Upload Benchmarks

We create two benchmarks:
- **MMLU-Pro** — for SFT models (chat-based multiple choice)
- **ARC-Challenge** — for CPT models (works in completion mode)

In [ ]:
os.makedirs("benchmarks", exist_ok=True)
with open("benchmarks/mmlu_pro.py", "w") as f:
    f.write("""
from inspect_ai import Task, task
from inspect_ai.dataset import Sample, hf_dataset
from inspect_ai.scorer import choice
from inspect_ai.solver import multiple_choice, generate

def record_to_sample(record):
    return Sample(input=record["question"], target=record["answer"], choices=record["options"])

@task
def mmlu_pro():
    dataset = hf_dataset("TIGER-Lab/MMLU-Pro", split="test", sample_fields=record_to_sample)
    return Task(dataset=dataset, solver=[multiple_choice(), generate()], scorer=choice())
""")
with open("benchmarks/arc_challenge.py", "w") as f:
    f.write("""
from inspect_ai import Task, task
from inspect_ai.dataset import Sample, hf_dataset
from inspect_ai.scorer import choice
from inspect_ai.solver import multiple_choice, generate

def record_to_sample(record):
    return Sample(input=record["question"], target=record["answerKey"], choices=record["choices"]["text"])

@task
def arc_challenge():
    dataset = hf_dataset(path="allenai/ai2_arc", name="ARC-Challenge", split="test", sample_fields=record_to_sample)
    return Task(dataset=dataset, solver=[multiple_choice(), generate()], scorer=choice())
""")
with open("benchmarks/requirements.txt", "w") as f:
    f.write("datasets>=2.21.0\n")
for root, _, files in os.walk("benchmarks"):
    for fn in files:
        s3.upload_file(os.path.join(root, fn), S3_BUCKET, os.path.join(root, fn))
print("\u2713 Benchmarks uploaded")

### Write the Eval Config

The config selects the benchmark and mode based on your model type:
- **SFT:** Runs MMLU-Pro in chat mode
- **CPT:** Runs ARC-Challenge with `completion_mode=True`

In [ ]:
if MODEL_TYPE == "sft":
    tasks = [{"name": "mmlu_pro", "limit": 5}]
    extra_args = []
else:
    tasks = [{"name": "arc_challenge", "limit": 5}]
    extra_args = ["-M", "completion_mode=True"]

config = {
    "inference_provider": {"sagemaker_endpoint": {"endpoint_name": ENDPOINT_NAME, "region": REGION}},
    "benchmarks": {"s3_path": f"s3://{S3_BUCKET}/benchmarks/", "tasks": tasks},
    "eval": {"max_connections": 4, "max_retries": 3, "timeout": 600, "decoding": {"temperature": 0.0, "max_tokens": 4096}},
    "output": {"s3_path": f"s3://{S3_BUCKET}/eval-results/"},
}
if extra_args:
    config["eval"]["extra_args"] = extra_args

os.makedirs("config", exist_ok=True)
with open("config/inspect_config.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)
s3.upload_file("config/inspect_config.yaml", S3_BUCKET, "config/inspect_config.yaml")
print(f"\u2713 Config uploaded ({MODEL_TYPE} mode)")
print("---")
print(yaml.dump(config, default_flow_style=False))

### (Optional) Enable MLflow Tracking

#### What is MLflow?

[Amazon SageMaker MLflow](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html) is an experiment tracking service that lets you compare evaluation results across multiple runs. When enabled, the container automatically logs:

- **Metrics** — accuracy scores for each benchmark
- **Parameters** — endpoint name, task name, model configuration
- **Artifacts** — the full .eval log files for detailed analysis

#### Why use it?

- Compare model performance across training checkpoints
- Track improvement over fine-tuning iterations
- Share results with your team via the MLflow UI

#### How it works

1. You create a tracking server (one-time setup, takes ~5 min)
2. Add the server ARN to your eval config
3. The container logs metrics after each benchmark completes
4. View results in the MLflow web UI


In [ ]:
# Create MLflow tracking server (skip if you already have one)
mlflow_name = "inspectlens-tracker"
try:
    sagemaker_client = boto3.client("sagemaker", region_name=REGION)
    sagemaker_client.create_mlflow_tracking_server(
        TrackingServerName=mlflow_name,
        ArtifactStoreUri=f"s3://{S3_BUCKET}/mlflow/",
        RoleArn=ROLE_ARN,
    )
    print(f"\u2713 Creating MLflow server: {mlflow_name} (takes 5-10 min)")
except sagemaker_client.exceptions.ClientError as e:
    if "already exists" in str(e) or "ResourceInUse" in str(e):
        print(f"\u2713 MLflow server exists: {mlflow_name}")
    else:
        print(f"\u26a0 MLflow creation failed: {e}")

# Get ARN and add to config
try:
    resp = sagemaker_client.describe_mlflow_tracking_server(TrackingServerName=mlflow_name)
    MLFLOW_ARN = resp["TrackingServerArn"]
    config["tracking"] = {
        "mlflow_tracking_arn": MLFLOW_ARN,
        "mlflow_experiment_name": "nova-eval",
        "mlflow_tracing": True,
        "mlflow_log_artifacts": True,
    }
    # Re-upload config with MLflow
    with open("config/inspect_config.yaml", "w") as f:
        yaml.dump(config, f, default_flow_style=False)
    s3.upload_file("config/inspect_config.yaml", S3_BUCKET, "config/inspect_config.yaml")
    print(f"\u2713 Config updated with MLflow tracking")
except Exception as e:
    print(f"\u26a0 Skipping MLflow: {e}")


### View MLflow Results (after eval completes)

Run this cell after the evaluation job finishes to get the MLflow UI link:


In [ ]:
# Get MLflow UI URL
try:
    resp = sagemaker_client.create_presigned_mlflow_tracking_server_url(
        TrackingServerName=mlflow_name,
    )
    print(f"MLflow UI: {resp['AuthorizedUrl']}")
except Exception as e:
    print(f"MLflow URL not available: {e}")


### Submit and Monitor

In [ ]:
trainer = ModelTrainer(
    training_image=IMAGE_URI, role=ROLE_ARN,
    compute=Compute(instance_type="ml.m5.large", instance_count=1, volume_size_in_gb=30),
    output_data_config=OutputDataConfig(s3_output_path=f"s3://{S3_BUCKET}/output/"),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=86400),
    base_job_name="inspect-eval-endpoint",
)
trainer.train(input_data_config=[InputData(channel_name="config", data_source=f"s3://{S3_BUCKET}/config/")], wait=False)
job_name = trainer._latest_training_job.training_job_name
print(f"\u2713 Job: {job_name}")
print(f"  https://{REGION}.console.aws.amazon.com/sagemaker/home?region={REGION}#/jobs/{job_name}")

# Monitor
while True:
    resp = sagemaker_client.describe_training_job(TrainingJobName=job_name)
    status = resp["TrainingJobStatus"]
    if status in ("Completed", "Failed", "Stopped"):
        print(f"\n\u2713 {status}")
        if status == "Failed":
            print(f"  Reason: {resp.get('FailureReason')}")
        break
    print(f"  {status} / {resp.get('SecondaryStatus', '')}...", end="\r")
    time.sleep(15)

### View Results

Download results from S3 and view the status summary. You can also explore results interactively:

- **Inspect AI Viewer** (from S3 directly): `inspect view --log-dir s3://YOUR_BUCKET/eval-results/JOB_NAME/eval_results/`
- **Inspect AI Viewer** (local): `inspect view --log-dir ./results/`
- **VS Code Extension**: Install the [Inspect AI extension](https://marketplace.visualstudio.com/items?itemName=aisi-inspect.inspect-ai) to browse results in your editor

> **If the job fails**, check:
> - The endpoint is `InService` and in the same region as the config
> - The IAM role has `sagemaker:InvokeEndpoint` permission
> - For CPT: ensure `completion_mode=True` is in extra_args


In [ ]:
os.makedirs("results", exist_ok=True)
resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=f"eval-results/{job_name}/eval_results/")
if "Contents" in resp:
    for obj in resp["Contents"]:
        s3.download_file(S3_BUCKET, obj["Key"], os.path.join("results", os.path.basename(obj["Key"])))
        print(f"  Downloaded: {os.path.basename(obj['Key'])}")

with open("results/_status.json") as f:
    st = json.load(f)
print(f"\nStatus: {st['status']}")
print(f"Passed: {st['passed_tasks']}")
print(f"Failed: {st['failed_tasks']}")

---

## Customization

### Increase parallelism

Match `max_connections` to your endpoint's `max_concurrency`:
```yaml
eval:
  max_connections: 16
```

### Run more samples

```yaml
tasks:
  - name: mmlu_pro   # Full dataset (~12K samples)
```

### Export as CSV

```yaml
output:
  s3_path: "s3://your-bucket/eval-results/"
  output_format: "csv"
```

## Cleanup

In [ ]:
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=S3_BUCKET):
    if "Contents" in page:
        s3.delete_objects(Bucket=S3_BUCKET, Delete={"Objects": [{"Key": o["Key"]} for o in page["Contents"]]})
s3.delete_bucket(Bucket=S3_BUCKET)
for arn in policies:
    iam.detach_role_policy(RoleName=ROLE_NAME, PolicyArn=arn)
iam.delete_role(RoleName=ROLE_NAME)
import shutil
for d in ["benchmarks", "config", "results"]:
    shutil.rmtree(d, ignore_errors=True)
print("\u2713 Cleaned up")
print("\nNote: Your SageMaker endpoint is still running. Delete it separately if no longer needed.")